# WAS Exam Practice Arena

Simulates a three-way accessibility testing conversation to help candidates prepare for the IAAP Web Accessibility Specialist exam. The first LLM generates a realistic WAS exam question covering a randomly selected content area from the [WAS Credential Content Outline](https://www.accessibilityassociation.org/was-credential-content-outline) — along with the question type (multiple choice, scenario-based, or code analysis) and the WCAG success criterion or standard being tested. The user enters their own answer, then the second LLM independently answers the same question with a detailed explanation and concrete examples. The third LLM acts as the examiner — it grades both the user's response and the LLM's response on a 0-10 scale using the same rubric, explains what each answer got right and wrong, identifies the exact criterion and content area being tested, and declares a winner. The full conversation including question, both answers, and the examiner's verdict is passed as JSON between turns so the examiner has complete context. The user can request a new question in the same content area to drill weak areas or ask for a random area to simulate full exam conditions.

## WAS Credential Content Outline

The exam covers three content areas weighted as follows:

**Content Area 1 — Creating Accessible Web Solutions (40%)**
- 1A: Guidelines, principles, and techniques for meeting success criteria (WCAG 2.x, WAI-ARIA, ATAG, EN 301 549)
- 1B: Applying guidelines and techniques to code (HTML, CSS, JavaScript, ARIA roles/states/properties, forms, images, multimedia)
- 1C: Accessible UX design and authoring practices

**Content Area 2 — Identifying Accessibility Issues in Web Solutions (40%)**
- 2A: Identifying issues using guidelines and success criteria
- 2B: Identifying issues using assistive technologies (JAWS, NVDA, VoiceOver, TalkBack, keyboard-only)
- 2C: Identifying issues using testing tools (automated scanners, manual checklists)

**Content Area 3 — Remediating Accessibility Issues in Web Solutions (20%)**
- 3A: Recommending and implementing solutions
- 3B: Prioritizing and planning remediation

In [1]:
# imports

import os
import requests
import json
import random
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, we can use the OpenAI python client
# Because Google has an endpoint compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


## Step 1: Content areas, question types, and conversation log

Defines `WAS_CONTENT_AREAS` and `WAS_DOMAINS` (the WAS Credential Content Outline sub-areas and weighted domains), `QUESTION_TYPES`, and an empty `conversation_log` list that will accumulate each round's question, answers, and verdict.

In [4]:
# Constants and data structures
WAS_CONTENT_AREAS = [
    "1A: Guidelines, Principles, and Techniques",    # Creating Accessible Web Solutions (40%)
    "1B: Applying Guidelines and Techniques to Code",
    "1C: Accessible UX Design and Authoring",
    "2A: Identifying Issues Using Guidelines",        # Identifying Accessibility Issues (40%)
    "2B: Identifying Issues Using Assistive Technologies",
    "2C: Identifying Issues Using Testing Tools",
    "3A: Recommending and Implementing Solutions",    # Remediating Issues (20%)
    "3B: Prioritizing and Planning Remediation",
]

WAS_DOMAINS = [
    "Content Area 1: Creating Accessible Web Solutions (40%)",
    "Content Area 2: Identifying Accessibility Issues (40%)",
    "Content Area 3: Remediating Accessibility Issues (20%)",
]

QUESTION_TYPES = ["multiple choice", "scenario-based", "code analysis"]

conversation_log = []

## Step 2: Question-setter (LLM #1)

`generate_question()` prompts GPT-4.1-mini to write a WAS exam question for a given (or random) content area and question type, returning it as a JSON dict with the question text, options (for multiple choice), and the WCAG criterion being tested.

In [5]:
question_setter_system_prompt = """You are an IAAP (International Association of Accessibility Professionals) \
Web Accessibility Specialist (WAS) exam question writer with deep expertise in WCAG 2.x, \
WAI-ARIA, ATAG, and EN 301 549.

Your task is to write one realistic WAS exam question for the specified content area and question type.

Rules:
- Match the difficulty and style of the real IAAP WAS exam.
- The WCAG success criterion or standard being tested must be real and precisely cited \
(e.g., "WCAG 2.1 SC 1.3.1 Info and Relationships (Level A)").
- For multiple choice: include exactly four options (A–D). Only one must be correct. \
Distractors must be plausible — common misconceptions, not obviously wrong answers.
- For scenario-based: describe a realistic web or application context before posing the question.
- For code analysis: include a short HTML, CSS, or JavaScript snippet and ask what the \
accessibility issue is or how to fix it.

Response format:
Respond with a single valid JSON object and nothing else — no markdown fences, no commentary \
before or after the JSON. Use exactly these keys:
{
  "content_area": "<the WAS content area name, verbatim>",
  "question_type": "<multiple choice | scenario-based | code analysis>",
  "wcag_criterion": "<primary WCAG SC or standard, e.g. WCAG 2.1 SC 1.4.3 Contrast (Minimum) (Level AA)>",
  "question": "<full question text>",
  "options": ["A. ...", "B. ...", "C. ...", "D. ..."]
}
Do not include the WCAG criterion in the question text — it should be cited only in the "wcag_criterion" field. 
The options list must be empty ([]) for scenario-based and code analysis questions."""

def generate_question(content_area=None):
    chosen_area = content_area or random.choices(WAS_DOMAINS, weights=[0.4, 0.4, 0.2])[0]
    chosen_type = random.choice(QUESTION_TYPES)
    messages = [{"role": "system", "content": question_setter_system_prompt},
        {"role": "user", "content": f"Write a {chosen_type} question for {chosen_area}."}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return json.loads(response.choices[0].message.content)

## Step 3: Display the question and collect an answer

`display_question()` renders the content area, question type, and question text (plus options for multiple choice) as Markdown, without revealing the WCAG criterion. The user's response is then captured with `input()`.

In [6]:
# Implement display_question(question_json).

def display_question(question_json):
    q = question_json
    # Header metadata
    lines = [
        f"**Content area:** {q['content_area']}",
        f"**Question type:** {q['question_type']}",
        "",
        q['question'],
    ]

    # Append options if multiple choice
    if q['question_type'] == "multiple choice":
        lines.append("")
        for option in q['options']:
            lines.append(option)

    markdown_string = "\n".join(lines)
    display(Markdown(markdown_string))

question = generate_question()
display_question(question)


**Content area:** Identifying Accessibility Issues
**Question type:** code analysis

Analyze the following HTML snippet and identify the primary accessibility issue it presents:

In [7]:
# Collect the user's answer.
user_answer = input("Your answer: ")

## Step 4: Model's independent answer (LLM #2)

`generate_model_answer()` sends the same question to Claude, which answers independently with its reasoning, the WCAG criterion it identifies, and a concrete example.

In [8]:
model_answerer_system_prompt = """You are a certified Web Accessibility Specialist (WAS) with ten years of \
hands-on experience auditing and remediating web applications for WCAG conformance. \
You use JAWS, NVDA, and VoiceOver daily and are deeply familiar with the ARIA Authoring Practices Guide.

You are answering a WAS exam question as a contestant. Answer as if this is a real exam — \
be clear and direct, without hedging.

Structure your response in exactly these four sections:

**Answer** — State your answer directly. For multiple choice: the letter and full option text. \
For open-ended: one direct sentence.

**Reasoning** — Explain why this is correct. Reference the underlying accessibility principle, \
not just the rule number. Be specific about who is harmed by a violation and how.

**WCAG criterion** — Identify the specific success criterion or standard the question is testing, \
with its number, name, and conformance level (A / AA / AAA). If multiple criteria apply, \
name the most directly relevant one.

**Concrete example** — Provide a short, practical example: a corrected HTML/ARIA snippet, \
a before/after code comparison, or a precise description of the screen reader announcement \
before and after the fix.

An examiner will grade your answer against the correct criterion. Naming the wrong success \
criterion costs points even if your reasoning is otherwise sound."""

def generate_model_answer(question_json):
    user_prompt = f"""
      Content area: {question_json['content_area']}
      Question type: {question_json['question_type']}
      {question_json['question']}
    """
    if question_json['question_type'] == "multiple choice":
        user_prompt += "\n\n" + "\n".join(question_json['options'])
    user_prompt += "\n\nAnswer this question as described in your instructions."
    messages = [{"role": "system", "content": model_answerer_system_prompt},
        {"role": "user", "content": user_prompt}]
    response = anthropic.chat.completions.create(model="claude-sonnet-4-6", messages=messages)
    return response.choices[0].message.content

model_anser = generate_model_answer(question)

## Step 5: Examiner (LLM #3)

`judge_round()` sends the question plus both the user's and the model's answers to Gemini, which grades both against the same rubric and returns a JSON verdict — scores, feedback, the real WCAG criterion, and a winner.

In [9]:
examiner_system_prompt = """You are a senior IAAP WAS certification examiner. You will receive a JSON object \
containing an exam question, a human candidate's answer, and an AI contestant's answer. \
Grade both answers using the same rubric — be rigorous, impartial, and specific.

Rubric (score each dimension separately, total out of 10):
- Accuracy (4 pts): Is the stated answer correct?
- Reasoning (3 pts): Is the explanation specific to the criterion, free of misconceptions, \
and clear about who is harmed and how?
- Criterion identification (2 pts): Is the WCAG success criterion correctly identified, \
with number, name, and level?
- Example quality (1 pt): Is the concrete example realistic, correct, and illustrative?

Scoring rules:
- Apply the identical rubric to both answers. Do not award credit to one answer for \
something the other answer also did.
- If both answers are correct, the one with more precise criterion identification and \
stronger reasoning wins.
- If both are wrong, the one closer to the correct criterion or principle wins. \
If equally wrong, declare a tie.
- The winner is determined by total score. Equal scores are a tie.

Response format:
Respond with a single valid JSON object and nothing else — no markdown fences, no text \
before or after the JSON object. Use exactly these keys:
{
  "wcag_criterion": "<correct WCAG SC or standard, with number, name, and level>",
  "was_content_area": "<WAS content area being tested>",
  "user_score": <integer 0–10>,
  "user_feedback": "<2–4 sentences: what the user got right, what they got wrong, what would have earned full marks>",
  "model_score": <integer 0–10>,
  "model_feedback": "<2–4 sentences: same structure as user_feedback>",
  "winner": "<user | model | tie>"
}"""

def judge_round(question_json, user_answer, model_answer):
    round_payload = {"question": question_json, "user_answer": user_answer,
        "model_answer": model_answer}
    messages = [{"role": "system", "content": examiner_system_prompt},
        {"role": "user", "content": json.dumps(round_payload)}]
    response = gemini.chat.completions.create(model="gemini-3.5-flash", messages=messages)
    return json.loads(response.choices[0].message.content)

verdict_json = judge_round(question, user_answer, model_anser)

## Step 6: Display the verdict

`display_verdict()` renders the examiner's verdict as Markdown: the real WCAG criterion and content area, a score/feedback table for the user and the model, and the winner.

In [10]:
# Implement display_verdict(verdict_json) and run one full round manually.

def display_verdict(verdict_json):
    v = verdict_json
    winner_labels = {
        "user": "You win this round!",
        "model": "The model wins this round!",
        "tie": "This round is a tie!",
    }
    winner_text = winner_labels.get(v["winner"], v["winner"])
    user_feedback = v["user_feedback"].replace("\n", " ")
    model_feedback = v["model_feedback"].replace("\n", " ")

    lines = [
        f"**WCAG criterion tested:** {v['wcag_criterion']}",
        f"**WAS content area:** {v['was_content_area']}",
        "",
        "| | Score | Feedback |",
        "|---|---|---|",
        f"| You | {v['user_score']}/10 | {user_feedback} |",
        f"| Model | {v['model_score']}/10 | {model_feedback} |",
        "",
        f"### {winner_text}",
    ]

    markdown_string = "\n".join(lines)
    display(Markdown(markdown_string))


display_verdict(verdict_json)


**WCAG criterion tested:** WCAG 2.1 SC 1.1.1 Non-text Content (Level A)
**WAS content area:** Identifying Accessibility Issues

| | Score | Feedback |
|---|---|---|
| You | 0/10 | You attempted to answer the question despite the HTML snippet being missing, guessing an issue ('no label in form elements') that does not align with the target WCAG 2.1 SC 1.1.1 (Non-text Content) criterion. You did not provide any reasoning, criterion identification, or concrete examples. To earn full marks, you would need to identify a SC 1.1.1 violation, explain who is harmed and how, cite the criterion name and level, and provide a before/after code example. |
| Model | 0/10 | You correctly pointed out that the HTML snippet was missing from the prompt, which prevented you from analyzing the code. However, because no actual answer, reasoning, criterion identification, or example was provided, no points can be awarded under the rubric. To earn points, the correct criterion (SC 1.1.1) and its associated details would need to be present. |

### This round is a tie!

## Optional Feature: Save and load sessions

`save_session()` writes `conversation_log` (with a timestamp) to a JSON file, and `load_session()` reads it back, so a session can be reviewed or resumed later.

In [12]:
def save_session(filename="was_session.json"):
    # Wrap conversation_log with a timestamp so each save records when it happened.
    data = {
        "saved_at": datetime.now().isoformat(),
        "rounds": conversation_log,
    }
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
        print(f"Saved {len(conversation_log)} round(s) to {filename}")

def load_session(filename="was_session.json"):
    with open(filename, "r") as f:
        return json.load(f)

## Optional Feature: Running scoreboard

`print_scoreboard()` tallies total points and win/tie counts across `conversation_log` and displays the results as a summary.

In [13]:
def print_scoreboard():
    user_total = sum(round["verdict"]["user_score"]  for round in conversation_log)
    model_total = sum(round["verdict"]["model_score"] for round in conversation_log)
    user_wins =sum(1 for round in conversation_log if round["verdict"]["winner"] == "user")
    model_wins = sum(1 for round in conversation_log if round["verdict"]["winner"] == "model")
    ties = sum(1 for round in conversation_log if round["verdict"]["winner"] == "tie")
    summary = f"""
    # Scoreboard

    ## Total Points
    - User: {user_total}
    - Model: {model_total}

    ## Wins/Ties
    - User: {user_wins}
    - Model: {model_wins}
    - Ties: {ties}
    """
    display(Markdown(summary))

## Optional Feature: Ask the examiner follow-up questions

`ask_examiner_followup()` replays the round's question, both answers, and the verdict back to the examiner along with a new question, and returns its plain-text reply.

In [ ]:
def ask_examiner_followup(round_data, followup_question):
    original_payload = {
        "question": round_data["question"],
        "user_answer": round_data["user_answer"],
        "model_answer": round_data["model_answer"],
    }
    messages = [
        {"role": "system", "content": examiner_system_prompt},
        {"role": "user", "content": json.dumps(original_payload)},
        {"role": "assistant", "content": json.dumps(round_data["verdict"])},
        {"role": "user", "content": followup_question},
    ]

    response = gemini.chat.completions.create(model="gemini-3.5-flash", messages=messages)
    return response.choices[0].message.content

## Step 7: The arena loop

`run_arena()` ties everything together: generate a question, collect both answers, judge the round, display and log the verdict, optionally ask a follow-up — and let the user drill a content area, go random, save/load a session, or quit.

In [ ]:
def run_arena():
    loaded_filename = None
    while True:
        choice = input(
            "New question — type a content area (e.g. '1A: Guidelines, Principles, and Techniques')\n"
            "to drill it, 'random' to simulate exam conditions, a filename ending in .json to load\n"
            "a previous session, or 'quit' to stop: "
        ).strip()

        if choice.lower() == "quit":
            while True:
                save_choice = input("Would you like to save this session? (y/n): ").strip().lower()
                if save_choice == "y":
                    save_session(loaded_filename or "was_session.json")
                    break
                elif save_choice == "n":
                    break
            print("Good luck on the exam!")
            break

        if choice.lower().endswith(".json"):
            try:
                loaded_data = load_session(choice)
            except FileNotFoundError:
                print(f"Couldn't find {choice} — check the filename and try again.")
                continue
            conversation_log.extend(loaded_data["rounds"])
            loaded_filename = choice
            print_scoreboard()
            continue

        content_area = choice if choice in WAS_CONTENT_AREAS else None
        question = generate_question(content_area=content_area)

        display_question(question)
        user_answer = input("Your answer: ")
        model_answer = generate_model_answer(question)
        verdict = judge_round(question, user_answer, model_answer)
        display_verdict(verdict)
        while True:
            has_followup = input("Would you like to ask a follow-up question about this verdict? (y/n): ").strip().lower()
            if has_followup == "y":
                followup_question = input("Enter your follow-up question about the verdict: ")
                round_data = {
                    "question": question,
                    "user_answer": user_answer,
                    "model_answer": model_answer,
                    "verdict": verdict,
                }
                reply = ask_examiner_followup(round_data, followup_question)
                display(Markdown(reply))
                break
            elif has_followup == "n":
                break

        conversation_log.append({
            "question": question,
            "user_answer": user_answer,
            "model_answer": model_answer,
            "verdict": verdict,
        })


## Step 8: Run the arena

Everything is wired up — start a session and practice for real.

In [15]:
run_arena()


**Content area:** Content Area 1: Creating Accessible Web Solutions (40%)
**Question type:** code analysis

Review the following HTML snippet used for navigation on a website:

<nav aria-label="Main navigation">
  <ul>
    <li><a href="home.html"><img src="home-icon.png" alt=""></a></li>
    <li><a href="about.html"><img src="about-icon.png"></a></li>
    <li><a href="contact.html"><img src="contact-icon.png" alt="Contact us"></a></li>
  </ul>
</nav>

What is the primary accessibility issue with this code?

**WCAG criterion tested:** WCAG 2.1 SC 1.1.1 Non-text Content (Level A)
**WAS content area:** Content Area 1: Creating Accessible Web Solutions (40%)

| | Score | Feedback |
|---|---|---|
| You | 0/10 | The user provided a single-character placeholder ('q') as an answer. No credit could be awarded for accuracy, reasoning, criterion identification, or example quality. A complete response addressing the accessibility of the HTML snippet was required. |
| Model | 10/10 | The model provided an exceptional, highly accurate response. It correctly identified that the first link has an empty alt attribute (leaving the link nameless) and the second link lacks an alt attribute entirely (causing filename fallback). The model accurately identified WCAG 1.1.1 Non-text Content (Level A) and WCAG 2.4.4 Link Purpose (Level A), explained the user impact flawlessly, and provided high-quality before/after code examples. |

### The model wins this round!

**Content area:** Content Area 2: Identifying Accessibility Issues (40%)
**Question type:** multiple choice

A website's main navigation menu uses light gray text (#CCCCCC) on a white background. An accessibility review reports insufficient contrast. Which of the following adjustments best resolves this issue?

A. Change the text color to dark gray (#333333)
B. Keep the text color but increase the font size from 12px to 14px
C. Add a background image behind the text to improve visibility
D. Use all uppercase letters for the menu items to improve legibility

**WCAG criterion tested:** WCAG 2.1 SC 1.4.3 Contrast (Minimum) (Level AA)
**WAS content area:** Content Area 2: Identifying Accessibility Issues (40%)

| | Score | Feedback |
|---|---|---|
| You | 0/10 | The user's response was a single letter ('y') which does not correspond to any valid option. No reasoning, criterion identification, or concrete examples were provided. To earn full marks, select the correct option (A) and provide a comprehensive explanation detailing the WCAG criterion, contrast calculations, and CSS examples. |
| Model | 10/10 | The model correctly identified Option A as the correct answer and provided a flawless explanation. It calculated the exact contrast ratios for both states, clearly refuted the incorrect options with technical accuracy, correctly cited the WCAG SC, and provided a realistic CSS code example showing the fix. |

### The model wins this round!

Good luck on the exam!
